# Animated coupling heatmaps, no mean-cos panel (block-mean cosine + block coupling)

Copy of the experiments_anim_3 notebook with the per-token `mean cos` coupling panel dropped (CKA + signed trace only). Every heatmap names its `cmap` explicitly, and the axis ticks are configurable (`tick_style` bare-number vs full label, `tick_every`, `tick_fontsize`). Progress-bar counts read in `log10` by default; `spectrum_anim.COUNT_STYLE` (or `count_style=` per call) switches to `pow10` / `sci`.


In [ ]:
import os, sys
import warnings
import importlib
if os.path.basename(os.getcwd()) == 'analysis':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
import utils.model_registry, utils.accessor
importlib.reload(utils.model_registry)   # deps first: reload(_lib) alone re-imports cached modules
importlib.reload(utils.accessor)
from analysis import experiments_lib as _lib
importlib.reload(_lib)
from analysis.experiments_lib import (
    build_hooks, get_ys, get_series_y, panel_palettes, smooth_spectrum,
    submatrix, block_mean_cos, model_name_options, YVAR_LABELS, XVAR_FNS)
# nanochat zero-inits c_proj: step-0 attn/mlp.out frames are all-zero; ylim is pinned
# globally so the log-autoscale warning on those frames is pure noise.
warnings.filterwarnings('ignore', message='Data has no positive values')

In [ ]:
# Config this notebook needs (kept out of the generic lib):
BLOCK_REPR = 'block_representations_all'         # all-layer cov runs (the 4 main models)
BLOCK_SAMPLES = 'block_representations_samples'  # samples runs (also carry 160m/410m)
SRC = {                                          # model -> source for spectra/means/eigvals
    'pythia-160m-deduped':  BLOCK_SAMPLES,
    'pythia-410m-deduped':  BLOCK_SAMPLES,
    'pythia-1b-deduped':    BLOCK_REPR,
    'pythia-6.9b-deduped':  BLOCK_REPR,
    'OLMo-2-0425-1B':       BLOCK_REPR,
    'OLMo-2-1124-7B':       BLOCK_REPR,
    'nanochat-d12':         'nanochat_samples',
}
SAMPLES_SRC = {m: 'nanochat_samples' if m == 'nanochat-d12' else BLOCK_SAMPLES for m in SRC}
measured_br = {m: list(range(L)) for m, L in [
    ('pythia-160m-deduped', 12), ('pythia-410m-deduped', 24),
    ('pythia-1b-deduped', 16),   ('pythia-6.9b-deduped', 32),
    ('OLMo-2-0425-1B', 16),      ('OLMo-2-1124-7B', 32),
    ('nanochat-d12', 12)]}
# sequential blocks (OLMo-2, nanochat) expose a distinct mlp.in; parallel Pythia aliases attn.in
has_mlp_in = lambda model: 'olmo' in model.lower() or 'nanochat' in model.lower()
HK = build_hooks()                     # hook-name -> (leaf, metric) table
def bnd(model, prefix, ms):
    return [(SRC[model], HK[f'{prefix}{l}_{m.upper()[:2]}'], f'{m} {l} {prefix}')
            for l in measured_br[model] for m in ms]
attn_in  = lambda model, ms=['Au']: bnd(model, 'AI', ms)
attn_out = lambda model, ms=['Au']: bnd(model, 'AO', ms)
mlp_in   = lambda model, ms=['Au']: bnd(model, 'MI', ms)
mlp_out  = lambda model, ms=['Au']: bnd(model, 'MO', ms)

In [ ]:
# Animated-spectra engine (analysis/spectrum_anim.py): inject this notebook's data
# backend, expose animate_spectra. The notebook drives animations via animate_spectra /
# anim_meancov / anim_blkres (no plot_spectrum needed here).
import analysis.spectrum_anim as sa
importlib.reload(sa)   # pick up engine edits without restarting the kernel
sa.configure(get_series_y=get_series_y, get_ys=get_ys,
             panel_palettes=panel_palettes, smooth_spectrum=smooth_spectrum,
             submatrix=submatrix, block_mean_cos=block_mean_cos,
             model_name_options=model_name_options, YVAR_LABELS=YVAR_LABELS, XVAR_FNS=XVAR_FNS)
# Progress-bar token count: 'log10' -> "9.09 log10 tokens", 'pow10' -> "10^9.09 tokens",
# 'sci' -> "1.23e+09 tokens". Per-call override: animate_spectra(..., count_style=...).
sa.COUNT_STYLE = 'log10'
animate_spectra = sa.animate_spectra


In [ ]:
from IPython.display import display

## Animated block-mean cosine heatmap (Exp 1.1)

Pairwise cosine between block-output means (block×block, ordered by depth: attn then mlp per layer), animated across checkpoints. Diagonal hidden. `pin_range=True` (default) holds one symmetric colour range across the whole sweep so colours stay comparable; `pin_range=False` rescales each frame for full contrast. `pair=(rows, cols)` label-substring-selects a sub-block pairing; each model runs the 4 pairings all×all, attn×attn, mlp×mlp, attn×mlp (`PAIRS`). `cmap` is explicit; `tick_style` / `tick_every` / `tick_fontsize` control the axis ticks (bare depth number, one tick per 3 layers, readable size).


In [ ]:
# Block×block cosine-of-means heatmap, animated across checkpoints (block_mean_cos per step).
# pin_range=True -> one symmetric colour range over the whole sweep (stable colorbar);
# pin_range=False -> rescale each frame for full per-frame contrast.
# pair=(rows, cols) -> submatrix by label substrings ('' = all); diagonal hidden iff rows == cols.
# cmap / tick_* go straight to the heatmap panel (spectrum_anim._matrix_spec):
# tick_style 'number' = bare depth index, 'label' = the full 'attn 3' label.
# save_dir (if given) also writes an mp4 in the background — it does NOT change the inline render.
PAIRS = [('', ''), ('attn', 'attn'), ('mlp', 'mlp'), ('attn', 'mlp')]
def _pair_tag(pair):
    return f"{pair[0] or 'all'}×{pair[1] or 'all'}"

def _tick_opts(pair, style, every, fontsize):
    # all×all axes interleave attn/mlp, so a step of n there alternates sub-block and skips
    # layers; double it so one labelled tick still means one layer.
    return {'tick_style': style, 'tick_fontsize': fontsize,
            'tick_every': every * (2 if pair == ('', '') else 1)}

def anim_blockmean_cos(model, save_dir=None, pin_range=True, pair=('', ''), cmap='coolwarm',
                       tick_style='number', tick_every=3, tick_fontsize=11, **kw):
    outs = [(SRC[model], HK[f'{p}{l}_AU'], f'{sub} {l}')          # attn then mlp, by depth
            for l in measured_br[model] for p, sub in [('AO', 'attn'), ('MO', 'mlp')]]
    tag = _pair_tag(pair)
    panel = ('acts_mean_vec', outs, [model],
             {'kind': 'heatmap', 'title': f'Block-mean cosine — {tag}', 'cmap': cmap,
              'per_frame': not pin_range, 'pair': pair,
              **_tick_opts(pair, tick_style, tick_every, tick_fontsize)})
    opts = dict(ncols=1, model=model, suptitle=f'Block-mean cosine — {model} — {tag}', **kw)
    save = f"{save_dir}/blockmean_cos_{model}_{tag.replace('×', 'x')}.mp4" if save_dir else None
    display(animate_spectra([panel], save=save, **opts))


In [ ]:
for pair in PAIRS:
    anim_blockmean_cos('pythia-160m-deduped', save_dir='analysis/figures/animations/no_mean_cos', pair=pair)

In [ ]:
for pair in PAIRS:
    anim_blockmean_cos('pythia-410m-deduped', save_dir='analysis/figures/animations/no_mean_cos', pair=pair)

In [ ]:
for pair in PAIRS:
    anim_blockmean_cos('pythia-1b-deduped', save_dir='analysis/figures/animations/no_mean_cos', pair=pair)

In [ ]:
for pair in PAIRS:
    anim_blockmean_cos('pythia-6.9b-deduped', save_dir='analysis/figures/animations/no_mean_cos', pair=pair)

In [ ]:
for pair in PAIRS:
    anim_blockmean_cos('OLMo-2-0425-1B', save_dir='analysis/figures/animations/no_mean_cos', pair=pair)

In [ ]:
for pair in PAIRS:
    anim_blockmean_cos('OLMo-2-1124-7B', save_dir='analysis/figures/animations/no_mean_cos', pair=pair)

In [ ]:
for pair in PAIRS:
    anim_blockmean_cos('nanochat-d12', save_dir='analysis/figures/animations/no_mean_cos', pair=pair)

## Animated block↔block coupling (samples run)

Pairwise block-output coupling matrices from the `block_representations_samples` runs (every layer), animated across checkpoints: **CKA** (shared subspace, unsigned, sequential `Blues` over [0,1]) and **signed trace** (net reinforce/cancel, energy-weighted, diverging `coolwarm` about 0). The per-token `mean cos` panel of experiments_anim_3 is dropped here. Covariance-level sequel to the block-mean cosine animation above; matrices are stored per checkpoint, so `kind='matrix'` reads them directly. Same 4-way `pair` pairings and the same tick controls as above.


In [ ]:
# Stored (M,M) coupling matrices, animated. cka is unsigned in [0,1] (static range, sequential
# cmap); signed trace is symmetric about 0 (diverging cmap, pin_range as above).
# pair=(rows, cols) -> submatrix by label substrings, applied to both panels.
def anim_block_coupling(model, save_dir=None, pin_range=True, pair=('', ''), src=None,
                        cka_cmap='Blues', signed_cmap='coolwarm',
                        tick_style='number', tick_every=3, tick_fontsize=11, **kw):
    src = src or SAMPLES_SRC[model]
    hook = ('', 'block_block_coupling')
    leaves = get_ys(src, model, hook, 'leaves')[0][0]
    labels = [l.removeprefix('blk').removesuffix('.out') for l in leaves]
    tag = _pair_tag(pair)
    ticks = _tick_opts(pair, tick_style, tick_every, tick_fontsize)
    panels = [(y, [(src, hook)], [model],
               {'kind': 'matrix', 'labels': labels, 'title': f'{t} — {tag}',
                'per_frame': not pin_range, 'pair': pair, **ticks, **o})
              for y, t, o in [('cka', 'CKA', {'dynamic': False, 'vmin': 0, 'vmax': 1,
                                              'cmap': cka_cmap}),
                              ('signed_trace', 'signed trace', {'cmap': signed_cmap})]]
    opts = dict(ncols=2, model=model, suptitle=f'Block↔block coupling — {model} — {tag}', **kw)
    save = f"{save_dir}/block_coupling_{model}_{tag.replace('×', 'x')}.mp4" if save_dir else None
    display(animate_spectra(panels, save=save, **opts))


In [ ]:
for pair in PAIRS:
    anim_block_coupling('pythia-160m-deduped', save_dir='analysis/figures/animations/no_mean_cos', pair=pair)

In [ ]:
for pair in PAIRS:
    anim_block_coupling('pythia-410m-deduped', save_dir='analysis/figures/animations/no_mean_cos', pair=pair)

In [ ]:
for pair in PAIRS:
    anim_block_coupling('pythia-1b-deduped', save_dir='analysis/figures/animations/no_mean_cos', pair=pair)

In [ ]:
for pair in PAIRS:
    anim_block_coupling('pythia-6.9b-deduped', save_dir='analysis/figures/animations/no_mean_cos', pair=pair)

In [ ]:
for pair in PAIRS:
    anim_block_coupling('OLMo-2-0425-1B', save_dir='analysis/figures/animations/no_mean_cos', pair=pair)

In [ ]:
for pair in PAIRS:
    anim_block_coupling('OLMo-2-1124-7B', save_dir='analysis/figures/animations/no_mean_cos', pair=pair)

In [ ]:
for pair in PAIRS:
    anim_block_coupling('nanochat-d12', save_dir='analysis/figures/animations/no_mean_cos', pair=pair)